# Healthcare MLOps Demo - Part 1: Data Ingestion & Pipeline Setup

**Duration:** ~30 minutes

## Overview
In this notebook, we'll set up the data foundation for our 30-day hospital readmission prediction model:

1. **Internal Stage Data Load** - Load synthetic healthcare data from internal stage
2. **Streams** - Set up CDC (Change Data Capture) for incremental processing
3. **Dynamic Tables** - Build bronze → silver transformation pipeline
4. **Tasks** - Create scheduled orchestration

## Business Context
Hospital readmissions within 30 days are a key quality metric. CMS penalizes hospitals with excess readmissions.
We'll build an ML pipeline to predict readmission risk and enable early intervention.

---
## Setup: Connect to Snowflake

In [ ]:
# Import required packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd

# Get active Snowflake session (automatically available in Snowflake Notebooks)
session = get_active_session()
print(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
# Set context
session.sql("USE DATABASE HEALTHCARE_MLOPS").collect()
session.sql("USE WAREHOUSE HEALTHCARE_ML_WH").collect()

print("Context set successfully!")

---
## 1. Data Ingestion from Internal Stage

We're loading synthetic healthcare data (Synthea-style) that includes:
- **Patients**: Demographics, insurance, location
- **Encounters**: Hospital visits, costs, reasons
- **Conditions**: Diagnoses with ICD-10 codes

> **Note**: Data is stored in an internal stage. The same approach works with external S3/Azure/GCS stages.

In [ ]:
# Check what files are in our internal stage
session.sql("""
    LIST @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE
""").show()

In [ ]:
# Load patients data from internal stage
session.sql("""
    COPY INTO HEALTHCARE_MLOPS.RAW.PATIENTS (
        PATIENT_ID, BIRTH_DATE, DEATH_DATE, SSN, DRIVERS_LICENSE, PASSPORT,
        PREFIX, FIRST_NAME, LAST_NAME, SUFFIX, MAIDEN_NAME, MARITAL_STATUS,
        RACE, ETHNICITY, GENDER, BIRTHPLACE, ADDRESS, CITY, STATE, COUNTY,
        ZIP, LAT, LON, HEALTHCARE_EXPENSES, HEALTHCARE_COVERAGE, INCOME
    )
    FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/patients/
    FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
    ON_ERROR = 'CONTINUE'
""").show()

print(f"Patients loaded: {session.table('HEALTHCARE_MLOPS.RAW.PATIENTS').count()} rows")

In [ ]:
# Load encounters data
session.sql("""
    COPY INTO HEALTHCARE_MLOPS.RAW.ENCOUNTERS (
        ENCOUNTER_ID, START_DATETIME, STOP_DATETIME, PATIENT_ID,
        ORGANIZATION_ID, PROVIDER_ID, PAYER_ID, ENCOUNTER_CLASS,
        CODE, DESCRIPTION, BASE_ENCOUNTER_COST, TOTAL_CLAIM_COST,
        PAYER_COVERAGE, REASON_CODE, REASON_DESCRIPTION
    )
    FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/encounters/
    FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
    ON_ERROR = 'CONTINUE'
""").show()

print(f"Encounters loaded: {session.table('HEALTHCARE_MLOPS.RAW.ENCOUNTERS').count()} rows")

In [ ]:
# Load conditions data
session.sql("""
    COPY INTO HEALTHCARE_MLOPS.RAW.CONDITIONS (
        START_DATE, STOP_DATE, PATIENT_ID, ENCOUNTER_ID, CODE, DESCRIPTION
    )
    FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/conditions/
    FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
    ON_ERROR = 'CONTINUE'
""").show()

print(f"Conditions loaded: {session.table('HEALTHCARE_MLOPS.RAW.CONDITIONS').count()} rows")

### Preview the Raw Data

In [ ]:
# Preview patients
session.table("HEALTHCARE_MLOPS.RAW.PATIENTS").select(
    "PATIENT_ID", "FIRST_NAME", "LAST_NAME", "BIRTH_DATE", 
    "GENDER", "CITY", "STATE"
).show(5)

In [ ]:
# Preview encounters with types
encounters_df = session.table("HEALTHCARE_MLOPS.RAW.ENCOUNTERS")
encounters_df.group_by("ENCOUNTER_CLASS").agg(
    F.count("*").alias("COUNT"),
    F.avg("TOTAL_CLAIM_COST").alias("AVG_COST")
).order_by(F.col("COUNT").desc()).show()

---
## 2. Create Streams for CDC (Change Data Capture)

Streams capture changes to tables, enabling incremental processing of new data.
This is essential for production ML pipelines where we need to process only new records.

In [ ]:
# Verify streams exist (created in setup script)
session.sql("SHOW STREAMS IN SCHEMA HEALTHCARE_MLOPS.RAW").show()

In [ ]:
# Check stream status - shows pending changes
print("Patients stream has data:", 
      session.sql("SELECT SYSTEM$STREAM_HAS_DATA('HEALTHCARE_MLOPS.RAW.PATIENTS_STREAM')").collect()[0][0])
print("Encounters stream has data:", 
      session.sql("SELECT SYSTEM$STREAM_HAS_DATA('HEALTHCARE_MLOPS.RAW.ENCOUNTERS_STREAM')").collect()[0][0])

---
## 3. Dynamic Tables: Bronze → Silver Pipeline

Dynamic Tables automatically maintain materialized views that refresh incrementally.
They're perfect for building data transformation pipelines.

### Pipeline Architecture:
```
RAW (Bronze) → CURATED (Silver) → FEATURE_STORE (Gold)
```

In [ ]:
# Dynamic Table 1: Curated Patients (Silver Layer)
# - Cleanse and standardize patient demographics
# - Calculate age and risk factors

session.sql("""
CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED
    TARGET_LAG = '1 hour'
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        PATIENT_ID,
        FIRST_NAME,
        LAST_NAME,
        BIRTH_DATE,
        DEATH_DATE,
        DATEDIFF('year', BIRTH_DATE, COALESCE(DEATH_DATE, CURRENT_DATE())) AS AGE,
        GENDER,
        RACE,
        ETHNICITY,
        MARITAL_STATUS,
        CITY,
        STATE,
        ZIP,
        LAT,
        LON,
        HEALTHCARE_EXPENSES,
        HEALTHCARE_COVERAGE,
        INCOME,
        -- Derived: Coverage ratio (financial risk indicator)
        CASE 
            WHEN HEALTHCARE_EXPENSES > 0 
            THEN HEALTHCARE_COVERAGE / HEALTHCARE_EXPENSES 
            ELSE 1.0 
        END AS COVERAGE_RATIO,
        -- Flag deceased patients
        CASE WHEN DEATH_DATE IS NOT NULL THEN TRUE ELSE FALSE END AS IS_DECEASED,
        LOADED_AT,
        CURRENT_TIMESTAMP() AS CURATED_AT
    FROM HEALTHCARE_MLOPS.RAW.PATIENTS
""").collect()

print("Created Dynamic Table: PATIENTS_CURATED")

In [ ]:
# Dynamic Table 2: Curated Encounters (Silver Layer)
# - Calculate length of stay
# - Categorize encounter types
# - Identify high-cost encounters

session.sql("""
CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
    TARGET_LAG = '1 hour'
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        ENCOUNTER_ID,
        PATIENT_ID,
        START_DATETIME,
        STOP_DATETIME,
        -- Length of stay in hours
        DATEDIFF('hour', START_DATETIME, STOP_DATETIME) AS LENGTH_OF_STAY_HOURS,
        -- Length of stay in days (for inpatient)
        DATEDIFF('day', START_DATETIME, STOP_DATETIME) AS LENGTH_OF_STAY_DAYS,
        ENCOUNTER_CLASS,
        -- Binary flag for inpatient
        CASE WHEN ENCOUNTER_CLASS = 'inpatient' THEN 1 ELSE 0 END AS IS_INPATIENT,
        CODE,
        DESCRIPTION,
        ORGANIZATION_ID,
        PROVIDER_ID,
        PAYER_ID,
        BASE_ENCOUNTER_COST,
        TOTAL_CLAIM_COST,
        PAYER_COVERAGE,
        -- Out of pocket cost
        (TOTAL_CLAIM_COST - PAYER_COVERAGE) AS OUT_OF_POCKET_COST,
        REASON_CODE,
        REASON_DESCRIPTION,
        -- High cost flag (top quartile)
        CASE WHEN TOTAL_CLAIM_COST > 15000 THEN 1 ELSE 0 END AS IS_HIGH_COST,
        LOADED_AT,
        CURRENT_TIMESTAMP() AS CURATED_AT
    FROM HEALTHCARE_MLOPS.RAW.ENCOUNTERS
""").collect()

print("Created Dynamic Table: ENCOUNTERS_CURATED")

In [ ]:
# Dynamic Table 3: Curated Conditions (Silver Layer)
# - Extract ICD-10 category
# - Flag chronic vs acute conditions

session.sql("""
CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.CONDITIONS_CURATED
    TARGET_LAG = '1 hour'
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        CONDITION_ID,
        PATIENT_ID,
        ENCOUNTER_ID,
        START_DATE,
        STOP_DATE,
        CODE AS ICD10_CODE,
        -- Extract ICD-10 category (first letter)
        SUBSTR(CODE, 1, 1) AS ICD10_CATEGORY,
        -- Extract ICD-10 subcategory (first 3 chars)
        SUBSTR(CODE, 1, 3) AS ICD10_SUBCATEGORY,
        DESCRIPTION,
        -- Duration in days (NULL if ongoing)
        DATEDIFF('day', START_DATE, COALESCE(STOP_DATE, CURRENT_DATE())) AS CONDITION_DURATION_DAYS,
        -- Flag chronic conditions (no end date)
        CASE WHEN STOP_DATE IS NULL THEN 1 ELSE 0 END AS IS_CHRONIC,
        -- Major condition categories based on ICD-10
        CASE 
            WHEN CODE LIKE 'I%' THEN 'Circulatory'
            WHEN CODE LIKE 'E%' THEN 'Endocrine'
            WHEN CODE LIKE 'J%' THEN 'Respiratory'
            WHEN CODE LIKE 'C%' THEN 'Neoplasm'
            WHEN CODE LIKE 'M%' THEN 'Musculoskeletal'
            WHEN CODE LIKE 'N%' THEN 'Genitourinary'
            WHEN CODE LIKE 'F%' THEN 'Mental'
            WHEN CODE LIKE 'K%' THEN 'Digestive'
            WHEN CODE LIKE 'G%' THEN 'Nervous'
            WHEN CODE LIKE 'S%' OR CODE LIKE 'T%' THEN 'Injury'
            ELSE 'Other'
        END AS CONDITION_CATEGORY,
        LOADED_AT,
        CURRENT_TIMESTAMP() AS CURATED_AT
    FROM HEALTHCARE_MLOPS.RAW.CONDITIONS
""").collect()

print("Created Dynamic Table: CONDITIONS_CURATED")

In [ ]:
# Verify Dynamic Tables were created and are refreshing
session.sql("""
    SELECT 
        NAME,
        SCHEDULING_STATE,
        TARGET_LAG,
        REFRESH_MODE
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLES())
    WHERE SCHEMA_NAME = 'CURATED'
""").show()

---
## 4. Generate Readmission Labels

For ML training, we need labels indicating whether each inpatient encounter 
resulted in a readmission within 30 days.

In [ ]:
# Create readmission labels by identifying 30-day readmissions
session.sql("""
CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.READMISSION_LABELS
    TARGET_LAG = '1 hour'
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    WITH inpatient_encounters AS (
        SELECT 
            ENCOUNTER_ID,
            PATIENT_ID,
            START_DATETIME,
            STOP_DATETIME,
            -- Get next encounter for same patient
            LEAD(START_DATETIME) OVER (
                PARTITION BY PATIENT_ID 
                ORDER BY START_DATETIME
            ) AS NEXT_ENCOUNTER_START,
            LEAD(ENCOUNTER_CLASS) OVER (
                PARTITION BY PATIENT_ID 
                ORDER BY START_DATETIME
            ) AS NEXT_ENCOUNTER_CLASS
        FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
        WHERE ENCOUNTER_CLASS = 'inpatient'
    )
    SELECT
        ENCOUNTER_ID,
        PATIENT_ID,
        STOP_DATETIME AS DISCHARGED_AT,
        -- Calculate days to next encounter
        DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) AS DAYS_TO_NEXT_ENCOUNTER,
        -- Readmission if next inpatient visit within 30 days
        CASE 
            WHEN NEXT_ENCOUNTER_CLASS = 'inpatient' 
                AND DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) <= 30 
                AND DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) >= 0
            THEN TRUE 
            ELSE FALSE 
        END AS READMITTED_WITHIN_30_DAYS,
        CURRENT_TIMESTAMP() AS LABEL_GENERATED_AT
    FROM inpatient_encounters
""").collect()

print("Created readmission labels table")

In [ ]:
# Check readmission rate
session.sql("""
    SELECT 
        COUNT(*) AS TOTAL_INPATIENT_ENCOUNTERS,
        SUM(CASE WHEN READMITTED_WITHIN_30_DAYS THEN 1 ELSE 0 END) AS READMISSIONS,
        ROUND(100.0 * SUM(CASE WHEN READMITTED_WITHIN_30_DAYS THEN 1 ELSE 0 END) / COUNT(*), 2) AS READMISSION_RATE_PCT
    FROM HEALTHCARE_MLOPS.CURATED.READMISSION_LABELS
""").show()

---
## 5. Create Scheduled Task for Pipeline Orchestration

Tasks can be used to orchestrate data pipeline operations on a schedule.
Here we create a task to refresh our curated layer and generate alerts for new high-risk patients.

In [ ]:
# Create a task to process new data and log statistics
session.sql("""
CREATE OR REPLACE TASK HEALTHCARE_MLOPS.RAW.DAILY_DATA_QUALITY_CHECK
    WAREHOUSE = HEALTHCARE_ML_WH
    SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
    COMMENT = 'Daily data quality check for healthcare data pipeline'
AS
BEGIN
    -- Log data quality metrics
    INSERT INTO HEALTHCARE_MLOPS.RAW.DATA_QUALITY_LOG
    SELECT
        CURRENT_TIMESTAMP() AS CHECK_TIME,
        'PATIENTS' AS TABLE_NAME,
        COUNT(*) AS ROW_COUNT,
        SUM(CASE WHEN BIRTH_DATE IS NULL THEN 1 ELSE 0 END) AS NULL_BIRTH_DATES,
        SUM(CASE WHEN PATIENT_ID IS NULL THEN 1 ELSE 0 END) AS NULL_IDS
    FROM HEALTHCARE_MLOPS.RAW.PATIENTS;
END
""").collect()

print("Created daily data quality check task")

In [ ]:
# Create data quality log table
session.sql("""
CREATE TABLE IF NOT EXISTS HEALTHCARE_MLOPS.RAW.DATA_QUALITY_LOG (
    CHECK_TIME TIMESTAMP_NTZ,
    TABLE_NAME VARCHAR(100),
    ROW_COUNT INT,
    NULL_BIRTH_DATES INT,
    NULL_IDS INT
)
""").collect()

print("Created data quality log table")

In [ ]:
# Show all tasks
session.sql("SHOW TASKS IN SCHEMA HEALTHCARE_MLOPS.RAW").show()

---
## Summary: Data Pipeline Architecture

We've built a complete data ingestion and transformation pipeline:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Internal Stage │────▶│   RAW Tables    │────▶│  Dynamic Tables │
│  (CSV Files)    │     │   (Bronze)      │     │   (Silver)      │
└─────────────────┘     └─────────────────┘     └─────────────────┘
                               │
                               ▼
                        ┌─────────────────┐
                        │    Streams      │
                        │    (CDC)        │
                        └─────────────────┘
```

### Key Objects Created:
- **3 RAW tables**: PATIENTS, ENCOUNTERS, CONDITIONS
- **3 Streams**: For CDC on each table
- **4 Dynamic Tables**: PATIENTS_CURATED, ENCOUNTERS_CURATED, CONDITIONS_CURATED, READMISSION_LABELS
- **1 Task**: Daily data quality check

### Next Steps:
Continue to **Notebook 2: Feature Engineering** to build the Feature Store.

In [ ]:
# Final verification - show all objects
print("=== RAW TABLES ===")
session.sql("SHOW TABLES IN SCHEMA HEALTHCARE_MLOPS.RAW").select('"name"', '"rows"').show()

print("\n=== DYNAMIC TABLES ===")
session.sql("SHOW DYNAMIC TABLES IN SCHEMA HEALTHCARE_MLOPS.CURATED").select('"name"', '"scheduling_state"').show()

print("\n=== STREAMS ===")
session.sql("SHOW STREAMS IN SCHEMA HEALTHCARE_MLOPS.RAW").select('"name"', '"stale"').show()